# 변경된 점
FeatureSASRec이 아니라 VectorSASRec 으로 변경

# 학습 하기 전 json 파일 점검

In [1]:
from pathlib import Path
import json, re
from datetime import datetime

BASE_DIR = Path.cwd()
RAW_VEC_PATH = BASE_DIR / "outputs_json" / "artwork_vector.json"
RAW_LOG_PATH = BASE_DIR / "outputs_json" / "user_logs.json"

OUT_VEC_NORM = BASE_DIR / "outputs_json" / "artwork_vector_norm.json"
OUT_LOG_NORM = BASE_DIR / "outputs_json" / "user_logs_norm.json"

print("RAW_VEC_PATH:", RAW_VEC_PATH)
print("RAW_LOG_PATH:", RAW_LOG_PATH)
print("OUT_VEC_NORM:", OUT_VEC_NORM)
print("OUT_LOG_NORM:", OUT_LOG_NORM)

RAW_VEC_PATH: /home/j-i14e107/Image_classification/outputs_json/artwork_vector.json
RAW_LOG_PATH: /home/j-i14e107/Image_classification/outputs_json/user_logs.json
OUT_VEC_NORM: /home/j-i14e107/Image_classification/outputs_json/artwork_vector_norm.json
OUT_LOG_NORM: /home/j-i14e107/Image_classification/outputs_json/user_logs_norm.json


In [2]:
ACTION_SET = {"VIEW", "LIKE", "STAY", "COMMENT", "REVIEW"}

def stem_artwork_id(x: str) -> str:
    """'category030_0006.png' -> 'category030_0006' 형태로 통일"""
    if x is None:
        return None
    s = str(x).strip()
    if not s:
        return None
    # 경로가 들어오면 파일명만
    s = s.split("/")[-1]
    # 확장자 제거
    if "." in s:
        s = ".".join(s.split(".")[:-1])
    return s

def parse_ts_if_possible(x):
    """timestamp 파싱(진짜 timestamp인 경우만). 실패하면 None."""
    if x is None:
        return None
    if isinstance(x, (int, float)):
        try:
            # ms 가능성
            if x > 1e12:
                return datetime.fromtimestamp(x / 1000.0).isoformat()
            return datetime.fromtimestamp(x).isoformat()
        except Exception:
            return None
    if not isinstance(x, str):
        return None
    s = x.strip()
    if not s:
        return None
    # timestamp 자리에 VIEW 같은 액션이 들어온 케이스 방지
    if s.upper() in ACTION_SET:
        return None
    try:
        s2 = s.replace("Z", "+00:00")
        return datetime.fromisoformat(s2).isoformat()
    except Exception:
        return None

def dump_jsonl(path: Path, rows_iter):
    path.parent.mkdir(parents=True, exist_ok=True)
    n = 0
    with path.open("w", encoding="utf-8") as f:
        for row in rows_iter:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            n += 1
    return n

In [3]:
import json
import numpy as np

# 1. 원본 데이터 로드
vec_raw = json.loads(RAW_VEC_PATH.read_text(encoding="utf-8"))

seen = set()
kept = 0
skipped = 0

# 2. 데이터 처리 제너레이터 (로직은 그대로 유지)
def iter_vec_norm():
    global kept, skipped
    for row in vec_raw:
        if not isinstance(row, dict):
            skipped += 1
            continue

        # stem_artwork_id 함수가 정의되어 있다고 가정합니다.
        aid = stem_artwork_id(row.get("artwork_id"))
        vec = row.get("artwork_vector")  # 원본 키

        if aid is None or vec is None:
            skipped += 1
            continue

        # 벡터 차원 체크(512 기대)
        v = np.asarray(vec, dtype=np.float32)
        if v.ndim != 1 or v.shape[0] != 512:
            skipped += 1
            continue

        # 중복 artwork_id는 첫 번째만 유지
        if aid in seen:
            skipped += 1
            continue
        seen.add(aid)

        out = dict(row)  # 기존 필드 유지
        out["artwork_id"] = aid
        out["vector"] = v.tolist()       # 새 키로 통일
        out.pop("artwork_vector", None)  # 원본 키 제거

        kept += 1
        yield out

# 3. [변경] 리스트로 변환 후 JSON 저장
final_data = list(iter_vec_norm())  # 제너레이터를 리스트로 변환

print(f"[VEC] Saving {len(final_data)} items to {OUT_VEC_NORM} (JSON format)...")

with open(OUT_VEC_NORM, "w", encoding="utf-8") as f:
    # ensure_ascii=False: 한글 깨짐 방지
    # indent=2: 보기 좋게 줄바꿈 및 들여쓰기
    json.dump(final_data, f, ensure_ascii=False, indent=2)

print(f"[VEC] kept={kept} skipped={skipped} unique_items={len(seen)}")

[VEC] Saving 27702 items to /home/j-i14e107/Image_classification/outputs_json/artwork_vector_norm.json (JSON format)...
[VEC] kept=27702 skipped=0 unique_items=27702


In [5]:
import json

# ==========================================
# 1. 로그 파일 로드 (JSONL 방식 처리)
# ==========================================
print(f"Loading logs from {RAW_LOG_PATH}...")

log_raw = []
with open(RAW_LOG_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            # 한 줄씩 읽어서 리스트에 추가
            log_raw.append(json.loads(line))
        except json.JSONDecodeError:
            continue

print(f"Loaded {len(log_raw)} raw logs.")

matched = 0
missed = 0

# 앞 단계에서 만든 vec_ids (또는 seen) 집합이 있다고 가정
# 만약 없다면 아래 줄 주석 해제하여 빈 집합으로 초기화 (테스트용)
# vec_ids = set() 

# ==========================================
# 2. 데이터 정제 및 변환
# ==========================================
processed_logs = []

# ACTION 집합 정의 (코드에 없어서 추가함, 필요시 수정)
ACTION_SET = {"VIEW", "LIKE", "STAY", "COMMENT", "REVIEW"}

for row in log_raw:
    if not isinstance(row, dict):
        missed += 1
        continue

    uid = row.get("member_id")
    # stem_artwork_id 함수가 있다고 가정
    aid = stem_artwork_id(row.get("artwork_id"))
    ts_or_act = row.get("timestamp")  # 기존 파일엔 여기에 VIEW가 들어있음

    if uid is None or aid is None:
        missed += 1
        continue

    # timestamp / action_type 분리 로직
    action_type = "VIEW" # 기본값
    timestamp = None

    # ts_or_act가 문자열이고 ACTION_SET에 있으면 action_type으로 간주
    if isinstance(ts_or_act, str) and ts_or_act.strip().upper() in ACTION_SET:
        action_type = ts_or_act.strip().upper()
    else:
        # 진짜 timestamp라면 파싱 (parse_ts_if_possible 함수 있다고 가정)
        # timestamp = parse_ts_if_possible(ts_or_act) 
        action_type = row.get("action_type") or "VIEW"

    # [수정 요청 사항 반영] timestamp 필드에 action_type을 넣음
    out = {
        "member_id": str(uid),
        "artwork_id": aid,
        "timestamp": action_type, 
    }

    # 벡터 매칭 확인
    if aid in vec_ids:
        matched += 1
    else:
        missed += 1

    processed_logs.append(out)

# ==========================================
# 3. 표준 JSON 파일로 저장 ([...])
# ==========================================
print(f"Saving {len(processed_logs)} logs to {OUT_LOG_NORM} (JSON Array)...")

with open(OUT_LOG_NORM, "w", encoding="utf-8") as f:
    # indent=2로 보기 좋게 저장
    json.dump(processed_logs, f, ensure_ascii=False, indent=2)

print(f"[LOG] matched_to_vec={matched} missed_to_vec={missed}")
if matched + missed > 0:
    print(f"[LOG] miss ratio={missed/(matched+missed):.4f}")

Loading logs from /home/j-i14e107/Image_classification/outputs_json/user_logs.json...
Loaded 3708695 raw logs.


NameError: name 'vec_ids' is not defined

# SASRec + TwoTower 학습 노트북

이 노트북은 **artwork_vector.json**(작품 임베딩)과 **train_user_logs.json/jsonl**(유저 로그)를 이용해서  
`VectorSASRec` + `TwoTowerAlign`(cosine/temperature scaling) 을 학습합니다.

- PAD index = 0, 실제 아이템은 1부터 시작
- 로그는 user별 시퀀스로 모아서 학습합니다.
- 평가: Leave-one-out (각 유저의 마지막 아이템 맞추기) HR/NDCG


In [6]:
# Cell 1) Config (경로/하이퍼파라미터만 여기서 수정)

from pathlib import Path

# 현재 노트북 실행 위치(=주피터의 working directory)를 기준으로 경로 잡기
BASE_DIR = Path.cwd()

# 입력 파일
VEC_PATH = str(BASE_DIR / "outputs_json" / "artwork_vector_norm.json")
LOG_PATH = str(BASE_DIR / "outputs_json" / "user_logs_3types.json")

# 출력 폴더
OUT_PTH_DIR = BASE_DIR / "outputs_pth"
OUT_PTH_DIR.mkdir(parents=True, exist_ok=True)

# 저장 파일
OUT_SASREC   = str(OUT_PTH_DIR / "BEST_SASRec_model.pth")
OUT_TWOTOWER = str(OUT_PTH_DIR / "BEST_BestRecommend_model.pth")

# ===== Model / Train =====
CLIP_DIM = 512  # item vector dim (e.g., CLIP)
HIDDEN_DIM = 512  # SASRec hidden dim
MAX_LEN = 200

BATCH_SIZE = 256
EPOCHS = 30
LR = 1e-3
SEED = 42

N_LAYERS = 2
N_HEADS = 4
DROPOUT = 0.1

LOGIT_SCALE = 20.0  # temperature scaling (bigger => sharper)

# ===== Action vocab =====
ACTION2ID = {"PAD": 0, "VIEW": 1, "LIKE": 2, "STAY": 3, "COMMENT": 4, "REVIEW": 5, "OTHER": 6}
N_ACTIONS = len(ACTION2ID)

# ===== User split =====
NUM_USERS = 20_000          # 유저 샘플링 상한(전체가 2만보다 적으면 있는만큼)
TRAIN_USER_RATIO = 0.8      # 유저 기준 80/20
MIN_LEN_FOR_TRAIN = 2       # train dataset(exclude_last_target=True) 만들려면 최소 3개 필요
MIN_LEN_FOR_EVAL  = 2       # eval은 최소 2개 필요

# ===== Cold / Normal / Heavy 기준 =====
COLD_MAX_LEN   = 5
NORMAL_MAX_LEN = 30
# heavy: > NORMAL_MAX_LEN

# "비율을 유지하면서 나누기"의 해석:
# - 기본값(ENFORCE_SEGMENT_RATIOS=False): 선택된 유저들의 원래 분포를 유지하면서 stratified 80/20 split
# - 필요하면 ENFORCE_SEGMENT_RATIOS=True로 켜고 TARGET_*로 "원하는 분포"에 맞춰 2만명을 샘플링할 수 있음
ENFORCE_SEGMENT_RATIOS = False
TARGET_COLD_RATIO   = 0.25
TARGET_NORMAL_RATIO = 0.55
TARGET_HEAVY_RATIO  = 0.20

# ===== Timestamp handling =====
USE_TIMESTAMP_SORT = True   # timestamp 있으면 user별 정렬로 순서 보장
LOGS_ARE_LATEST_FIRST = True  # 파일이 최신->과거 순서면 True (timestamp 정렬 사용시 큰 의미 없음)

# ===== MLflow =====
MLFLOW_ON = True
MLFLOW_TRACKING_URI = f"file:{(Path.cwd()/'mlruns').as_posix()}"
# 나중에 터미널에서 mlruns 파일을 받은 뒤 git bash에서
# mlflow ui --backend-store-uri mlruns --port 8080
# 를 쳐보면 기록 확인이 가능
MLFLOW_EXPERIMENT = "SASRec_TwoTower_Action_Stratified"
MLFLOW_RUN_NAME = None  # None이면 자동

print("LOG_PATH:", LOG_PATH)
print("VEC_PATH:", VEC_PATH)
print("OUT_SASREC:", OUT_SASREC)
print("OUT_TWOTOWER:", OUT_TWOTOWER)
print("MLFLOW_TRACKING_URI:", MLFLOW_TRACKING_URI if MLFLOW_ON else "(off)")

LOG_PATH: /home/j-i14e107/Image_classification/outputs_json/user_logs_3types.json
VEC_PATH: /home/j-i14e107/Image_classification/outputs_json/artwork_vector_norm.json
OUT_SASREC: /home/j-i14e107/Image_classification/outputs_pth/BEST_SASRec_model.pth
OUT_TWOTOWER: /home/j-i14e107/Image_classification/outputs_pth/BEST_BestRecommend_model.pth
MLFLOW_TRACKING_URI: file:/home/j-i14e107/Image_classification/mlruns


In [7]:
# Cell 2) Imports
import os, json, random
from pathlib import Path
from datetime import datetime

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

# MLflow
import mlflow

In [8]:
# Cell 3) Utils

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def read_json_or_jsonl(path: str):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {path}")
    txt = p.read_text(encoding="utf-8").strip()
    if not txt:
        raise ValueError(f"Empty file: {path}")

    # jsonl 감지
    if txt[0] == "{" and "\n" in txt:
        lines = [ln.strip() for ln in txt.splitlines() if ln.strip()]
        ok_jsonl = all(ln.startswith("{") and ln.endswith("}") for ln in lines[: min(5, len(lines))])
        if ok_jsonl:
            return [json.loads(ln) for ln in lines]
    return json.loads(txt)

def _stem_id(x) -> str:
    return Path(str(x)).stem

def _parse_ts(x):
    """가능하면 datetime으로 파싱. 실패하면 None."""
    if x is None:
        return None
    if isinstance(x, (int, float)):
        try:
            if x > 1e12:  # ms
                return datetime.fromtimestamp(x / 1000.0)
            return datetime.fromtimestamp(x)
        except Exception:
            return None
    if not isinstance(x, str):
        return None
    s = x.strip()
    if not s:
        return None
    try:
        s2 = s.replace("Z", "+00:00")
        return datetime.fromisoformat(s2)
    except Exception:
        return None

def segment_by_len(L: int) -> str:
    if L <= COLD_MAX_LEN:
        return "cold"
    elif L <= NORMAL_MAX_LEN:
        return "normal"
    else:
        return "heavy"

def split_users_stratified(users, user2seg, train_ratio=0.8, seed=42):
    """세그먼트 분포를 유지하면서(층화) 유저를 train/infer로 split"""
    rng = np.random.default_rng(seed)
    per = {"cold": [], "normal": [], "heavy": []}
    for u in users:
        per[user2seg[u]].append(u)

    train, infer = [], []
    for seg, lst in per.items():
        lst = lst.copy()
        rng.shuffle(lst)
        n_train = int(round(len(lst) * train_ratio))
        train.extend(lst[:n_train])
        infer.extend(lst[n_train:])

    rng.shuffle(train)
    rng.shuffle(infer)

    counts = {seg: len(lst) for seg, lst in per.items()}
    return train, infer, counts

def pick_users_with_optional_ratios(eligible_users, user2seg, num_users, seed=42):
    """
    ENFORCE_SEGMENT_RATIOS=False: eligible_users 중 무작위로 num_users 선택
    ENFORCE_SEGMENT_RATIOS=True : TARGET_* 비율에 맞춰 세그먼트별로 샘플링 (가능한 범위 내)
    """
    rng = np.random.default_rng(seed)

    if len(eligible_users) <= num_users:
        chosen = eligible_users.copy()
        rng.shuffle(chosen)
        return chosen

    if not ENFORCE_SEGMENT_RATIOS:
        chosen = rng.choice(eligible_users, size=num_users, replace=False).tolist()
        rng.shuffle(chosen)
        return chosen

    # enforce ratios
    per = {"cold": [], "normal": [], "heavy": []}
    for u in eligible_users:
        per[user2seg[u]].append(u)
    for seg in per:
        rng.shuffle(per[seg])

    target = {
        "cold": int(round(num_users * TARGET_COLD_RATIO)),
        "normal": int(round(num_users * TARGET_NORMAL_RATIO)),
        "heavy": int(round(num_users * TARGET_HEAVY_RATIO)),
    }

    # 보정(합이 딱 num_users 되도록)
    s = sum(target.values())
    if s != num_users:
        # normal에 몰아넣어 보정
        target["normal"] += (num_users - s)

    chosen = []
    shortage = 0

    for seg in ["cold", "normal", "heavy"]:
        take = min(target[seg], len(per[seg]))
        chosen.extend(per[seg][:take])
        if take < target[seg]:
            shortage += (target[seg] - take)

    if shortage > 0:
        # 남는 세그먼트에서 추가로 채우기
        rest = []
        for seg in ["cold", "normal", "heavy"]:
            rest.extend(per[seg][target[seg]:])  # target만큼 뽑고 남은 것들
        rng.shuffle(rest)
        chosen.extend(rest[:shortage])

    # 최종 개수 맞추기(초과 시 잘라내기)
    rng.shuffle(chosen)
    chosen = chosen[:num_users]
    return chosen

def ratio_dict(counts: dict):
    total = sum(counts.values())
    if total <= 0:
        return {k: 0.0 for k in counts}
    return {k: float(v) / float(total) for k, v in counts.items()}

def print_split_stats(title: str, users, user2seg):
    c = {"cold": 0, "normal": 0, "heavy": 0}
    for u in users:
        c[user2seg[u]] += 1
    r = ratio_dict(c)
    print(f"[{title}] n={sum(c.values())} | cold={c['cold']} ({r['cold']:.3f}) "
          f"normal={c['normal']} ({r['normal']:.3f}) heavy={c['heavy']} ({r['heavy']:.3f})")
    return c, r

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [9]:
# Cell 4) Models (VectorSASRec + TwoTower)
# - num_items에 의존하는 nn.Embedding(num_items, ...)을 제거하고
# - 런타임에 주입되는 item_vectors(artwork_vector)를 F.embedding으로 lookup해서 사용
# - 학습 파라미터는 item_in_proj + transformer + action/pos emb + TwoTower projection만 저장됨
# => 서비스에서 아이템 수가 바뀌어도(pth 로드시) size mismatch가 나지 않음

class VectorSASRec(nn.Module):
    def __init__(
        self,
        clip_dim: int,
        hidden_dim: int,
        num_actions: int,
        n_layers: int = 2,
        n_heads: int = 4,
        dropout: float = 0.1,
        maxlen: int = 200,
    ):
        super().__init__()
        self.clip_dim = clip_dim
        self.hidden_dim = hidden_dim
        self.maxlen = maxlen

        # ✅ item vector(clip_dim) -> hidden_dim projection (trainable)
        self.item_in_proj = nn.Linear(clip_dim, hidden_dim, bias=False)

        # ✅ action / position embedding in hidden_dim space
        self.act_emb = nn.Embedding(num_actions, hidden_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(maxlen, hidden_dim)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            dim_feedforward=4 * hidden_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def item_base(self, item_vectors: torch.Tensor) -> torch.Tensor:
        """(N, clip_dim) -> (N, hidden_dim)"""
        return self.item_in_proj(item_vectors)

    def forward(self, item_ids: torch.Tensor, action_ids: torch.Tensor, item_vectors: torch.Tensor):
        """
        item_ids:   (B,S) padding=0
        action_ids: (B,S) padding=0
        item_vectors: (N, clip_dim)  (0번은 PAD=0벡터 권장)
        return: (B,S,hidden_dim)
        """
        B, S = item_ids.shape
        assert action_ids.shape == item_ids.shape
        assert S <= self.maxlen, f"seq len {S} > maxlen {self.maxlen}"
        assert item_vectors.dim() == 2 and item_vectors.shape[1] == self.clip_dim, "item_vectors shape mismatch"

        # ✅ 런타임 item_vectors에서 lookup (Embedding 테이블 파라미터 없음)
        v = F.embedding(item_ids, item_vectors)      # (B,S,clip_dim)
        x = self.item_in_proj(v) + self.act_emb(action_ids)  # (B,S,hidden_dim)

        pos = torch.arange(S, device=item_ids.device).unsqueeze(0).expand(B, S)
        x = x + self.pos_emb(pos)
        x = self.dropout(x)

        pad_mask = (item_ids == 0)
        x = self.encoder(x, src_key_padding_mask=pad_mask)
        return x

class TwoTowerAlign(nn.Module):
    def __init__(self, dim=512, dropout=0.1):
        super().__init__()
        self.user_proj = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )
        self.item_proj = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )


In [10]:
# Cell 5) Data loaders (patched for new schema)
# - Vectors: accept keys 'vector' or 'artwork_vector' or 'embedding'
# - Logs: accept
#   (A) Flat event logs (JSONL/JSON array): {"member_id","artwork_id","action_type","timestamp"?}
#   (B) Grouped per-user logs (JSONL/JSON array):
#       {"member_id": "...", "timestamp": [{"artwork_id":"...","timestamp":"VIEW"}, ...]}
#     where inner "timestamp" == action_type and the LIST ORDER may be latest-first.

def load_item_vectors(vec_json_path: str, expected_dim: int = 512):
    data = read_json_or_jsonl(vec_json_path)

    artwork2idx = {"<PAD>": 0}
    matrix_list = [np.zeros(expected_dim, dtype=np.float32)]

    def add_one(aid, vec):
        if aid is None or vec is None:
            return
        aid = _stem_id(aid)
        if aid in artwork2idx:
            return
        v = np.asarray(vec, dtype=np.float32)
        if v.ndim != 1 or v.shape[0] != expected_dim:
            return
        artwork2idx[aid] = len(matrix_list)
        matrix_list.append(v)

    if isinstance(data, dict):
        for k, v in data.items():
            add_one(k, v)
    elif isinstance(data, list):
        for row in data:
            if not isinstance(row, dict):
                continue
            aid = row.get("artwork_id") or row.get("id") or row.get("item_id")
            vec = row.get("vector") or row.get("artwork_vector") or row.get("embedding")
            add_one(aid, vec)
    else:
        raise ValueError("Unknown vector json format")

    mat = np.stack(matrix_list, axis=0)  # (N, D)
    return artwork2idx, torch.tensor(mat, dtype=torch.float32)

def _get_action_str_from_any(obj: dict, default="VIEW") -> str:
    if not isinstance(obj, dict):
        return default
    for k in ("action_type", "action", "event", "type", "timestamp"):
        v = obj.get(k)
        if isinstance(v, str) and v.strip():
            s = v.strip().upper()
            if s in ACTION_SET:
                return s
    return default

def _iter_flat_events(logs):
    # logs is list[dict]
    for log in logs:
        if not isinstance(log, dict):
            continue
        uid = log.get("member_id")
        aid = log.get("artwork_id")
        if uid is None or aid is None:
            continue
        yield str(uid), _stem_id(aid), _get_action_str_from_any(log), log.get("timestamp")

def _iter_grouped_events(grouped_rows):
    # grouped_rows is list[dict] with {"member_id", "timestamp":[{artwork_id,timestamp(action)}]}
    for row in grouped_rows:
        if not isinstance(row, dict):
            continue
        uid = row.get("member_id")
        seq = row.get("timestamp")
        if uid is None or not isinstance(seq, list):
            continue
        uid = str(uid)
        # seq order can be latest-first; we will handle outside
        for idx, ev in enumerate(seq):
            if not isinstance(ev, dict):
                continue
            aid = ev.get("artwork_id")
            if aid is None:
                continue
            act = _get_action_str_from_any(ev, default="VIEW")  # inner timestamp == action
            yield uid, _stem_id(aid), act, None, idx  # idx used only for stable ordering

def load_user_sequences(
    log_path: str,
    artwork2idx: dict,
    logs_are_latest_first: bool = True,
    use_timestamp_sort: bool = False,
):
    """
    Returns:
      - user_item_seq: {uid: [item_idx, ...]}  (chronological: past -> recent)
      - user_act_seq : {uid: [act_id , ...]}
      - missed: int
    """
    raw = read_json_or_jsonl(log_path)
    if not isinstance(raw, list):
        raise ValueError("Log must be JSON array or JSONL list")

    # Detect grouped schema (member_id + timestamp(list))
    is_grouped = False
    if len(raw) > 0 and isinstance(raw[0], dict) and isinstance(raw[0].get("timestamp"), list) and "artwork_id" not in raw[0]:
        is_grouped = True

    tmp = {}  # uid -> list of tuples for ordering
    missed = 0

    if not is_grouped:
        # Flat events
        for uid, aid, act, ts in _iter_flat_events(raw):
            if aid not in artwork2idx:
                missed += 1
                continue
            # timestamp sort optional
            dts = _parse_ts(ts) if use_timestamp_sort else None
            tmp.setdefault(uid, []).append((dts, aid, act))
    else:
        # Grouped events (no real time). Preserve list order + reverse if needed.
        for uid, aid, act, _ts, idx in _iter_grouped_events(raw):
            if aid not in artwork2idx:
                missed += 1
                continue
            # For grouped logs, we use idx for stable ordering
            tmp.setdefault(uid, []).append((idx, aid, act))

    # Build sequences
    user_item_seq = {}
    user_act_seq = {}
    for uid, rows in tmp.items():
        if not rows:
            continue

        if not is_grouped and use_timestamp_sort:
            rows = sorted(rows, key=lambda x: (x[0] is None, x[0]))  # None last
            if logs_are_latest_first:
                rows = list(reversed(rows))
        elif is_grouped:
            # rows are in given order by idx
            rows = sorted(rows, key=lambda x: x[0])
            if logs_are_latest_first:
                # input order is latest-first, so reverse to chronological
                rows = list(reversed(rows))
        else:
            # no timestamp sort; assume file order already matches logs_are_latest_first flag
            if logs_are_latest_first:
                rows = list(reversed(rows))

        items = []
        acts = []
        for _, aid, act in rows:
            items.append(artwork2idx[aid])
            acts.append(ACTION2ID.get(act, ACTION2ID["VIEW"]))

        user_item_seq[uid] = items
        user_act_seq[uid] = acts

    return user_item_seq, user_act_seq, missed

In [11]:
# Cell 6) Dataset (누수 방지 + action 포함)

class SASRecDataset(Dataset):
    def __init__(self, user_item_seq, user_act_seq, maxlen=200, exclude_last_target=True):
        """
        exclude_last_target=True면:
          - 학습에서 "마지막 아이템"은 target에 포함하지 않음 (평가/추론용으로 남김)
          - input  = seq[:-2]
          - target = seq[1:-1]
        """
        self.samples = []

        for uid, seq in user_item_seq.items():
            acts = user_act_seq.get(uid)
            if acts is None or len(seq) != len(acts):
                continue

            if exclude_last_target:
                if len(seq) < 3:
                    continue
                input_items = seq[:-2]
                input_acts  = acts[:-2]
                target_items = seq[1:-1]
            else:
                if len(seq) < 2:
                    continue
                input_items = seq[:-1]
                input_acts  = acts[:-1]
                target_items = seq[1:]

            if len(input_items) > maxlen:
                input_items  = input_items[-maxlen:]
                input_acts   = input_acts[-maxlen:]
                target_items = target_items[-maxlen:]

            pad_len = maxlen - len(input_items)
            input_items  = [0] * pad_len + input_items
            input_acts   = [0] * pad_len + input_acts
            target_items = [0] * pad_len + target_items

            self.samples.append((
                torch.tensor(input_items, dtype=torch.long),
                torch.tensor(input_acts, dtype=torch.long),
                torch.tensor(target_items, dtype=torch.long),
            ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [12]:
# Cell 7) Evaluation (전체 + 세그먼트별)

@torch.no_grad()
def evaluate_valid_action(
    sas_model, tt_model,
    user_item_seq, user_act_seq,
    all_item_vecs,   # (N,D) item vectors (trainable weight 권장)
    maxlen=200,
    device="cuda",
    k1=10, k2=20,
):
    sas_model.eval()
    tt_model.eval()

    all_item_vecs = all_item_vecs.to(device)

    base_items = sas_model.item_base(all_item_vecs)
    all_items_proj = tt_model.item_proj(base_items)
    all_items_proj = F.normalize(all_items_proj, p=2, dim=-1)

    HR_10, HR_20, NDCG_10, NDCG_20 = [], [], [], []

    users = [u for u in user_item_seq.keys() if len(user_item_seq[u]) >= MIN_LEN_FOR_EVAL]
    for uid in tqdm(users, desc="eval(users)", leave=False):
        seq = user_item_seq[uid]
        acts = user_act_seq[uid]

        input_seq = seq[:-1]
        input_act = acts[:-1]
        target_item = seq[-1]

        if len(input_seq) > maxlen:
            input_seq = input_seq[-maxlen:]
            input_act = input_act[-maxlen:]

        pad_len = maxlen - len(input_seq)
        item_tensor = torch.tensor(([0] * pad_len + input_seq), device=device).unsqueeze(0)
        act_tensor  = torch.tensor(([0] * pad_len + input_act), device=device).unsqueeze(0)

        sas_out = sas_model(item_tensor, act_tensor, all_item_vecs)
        last_emb = sas_out[:, -1, :]

        user_vec = tt_model.user_proj(last_emb)
        user_vec = F.normalize(user_vec, p=2, dim=-1)

        scores = torch.matmul(user_vec, all_items_proj.T).squeeze(0)
        scores[0] = -1e9

        _, top_indices = torch.topk(scores, k=k2)
        top_indices = top_indices.detach().cpu().numpy()

        hit_10 = (target_item in top_indices[:k1])
        hit_20 = (target_item in top_indices[:k2])

        HR_10.append(1.0 if hit_10 else 0.0)
        HR_20.append(1.0 if hit_20 else 0.0)

        ndcg_10 = 0.0
        ndcg_20 = 0.0
        if hit_10:
            rank = np.where(top_indices[:k1] == target_item)[0][0]
            ndcg_10 = 1.0 / np.log2(rank + 2)
        if hit_20:
            rank = np.where(top_indices[:k2] == target_item)[0][0]
            ndcg_20 = 1.0 / np.log2(rank + 2)

        NDCG_10.append(ndcg_10)
        NDCG_20.append(ndcg_20)

    def _mean(x): 
        return float(np.mean(x)) if len(x) else 0.0

    return _mean(HR_10), _mean(HR_20), _mean(NDCG_10), _mean(NDCG_20)

@torch.no_grad()
def evaluate_by_segment(
    sas_model, tt_model,
    infer_item, infer_act,
    user2seg,
    all_item_vecs,
    maxlen=200, device="cuda",
):
    """infer 유저를 세그먼트별로 나눠 HR/NDCG를 각각 계산"""
    out = {}
    for seg in ["cold", "normal", "heavy"]:
        sub_users = [u for u in infer_item.keys() if user2seg.get(u) == seg]
        sub_item = {u: infer_item[u] for u in sub_users}
        sub_act  = {u: infer_act[u]  for u in sub_users}

        hr10, hr20, ndcg10, ndcg20 = evaluate_valid_action(
            sas_model, tt_model,
            sub_item, sub_act,
            all_item_vecs,
            maxlen=maxlen,
            device=device
        )
        out[seg] = {"hr10": hr10, "hr20": hr20, "ndcg10": ndcg10, "ndcg20": ndcg20, "n_users": len(sub_users)}
    return out

In [13]:
# DEBUG CELL) 파일/스키마/매칭 진단

from collections import Counter

# 1) 벡터 파일 구조 확인
vec_raw = read_json_or_jsonl(VEC_PATH)
print("VEC type:", type(vec_raw))
if isinstance(vec_raw, dict):
    print("VEC dict keys sample:", list(vec_raw.keys())[:5])
elif isinstance(vec_raw, list):
    print("VEC list len:", len(vec_raw))
    if len(vec_raw) > 0 and isinstance(vec_raw[0], dict):
        print("VEC[0] keys:", list(vec_raw[0].keys()))
        print("VEC[0] sample:", {k: vec_raw[0].get(k) for k in list(vec_raw[0].keys())[:6]})
    else:
        print("VEC[0] type:", type(vec_raw[0]) if len(vec_raw)>0 else None)

# 2) 로그 파일 구조 확인
log_raw = read_json_or_jsonl(LOG_PATH)
print("\nLOG type:", type(log_raw))
print("LOG len:", len(log_raw) if isinstance(log_raw, list) else "not-list")
if isinstance(log_raw, list) and len(log_raw) > 0 and isinstance(log_raw[0], dict):
    print("LOG[0] keys:", list(log_raw[0].keys()))
    print("LOG[0] sample:", {k: log_raw[0].get(k) for k in list(log_raw[0].keys())[:8]})

# 3) 벡터 로드 후 매칭률 확인
artwork2idx, item_mat = load_item_vectors(VEC_PATH, expected_dim=HIDDEN_DIM)
user_item_seq, user_act_seq, missed = load_user_sequences(
    LOG_PATH, artwork2idx,
    logs_are_latest_first=LOGS_ARE_LATEST_FIRST,
    use_timestamp_sort=USE_TIMESTAMP_SORT
)

lens = [len(v) for v in user_item_seq.values()]
print("\nitem_mat:", tuple(item_mat.shape), "num_items:", len(artwork2idx)-1)
print("num_users:", len(user_item_seq), "missed_logs:", missed)

if len(lens):
    print("user seq len stats:",
          "min=", min(lens), "p50=", int(np.median(lens)), "p90=", int(np.quantile(lens, 0.9)), "max=", max(lens))
    cnt = Counter(lens)
    print("len==0:", cnt.get(0,0), "len==1:", cnt.get(1,0), "len==2:", cnt.get(2,0), "len>=3:", sum(v for k,v in cnt.items() if k>=3))
else:
    print("No user sequences built at all.")

VEC type: <class 'list'>
VEC list len: 27702
VEC[0] keys: ['artist_id', 'artwork_id', 'image_path', 'matched_key', 'clip_primary_label', 'clip_primary_score', 'clip_genres', 'clip_scores', 'vector']
VEC[0] sample: {'artist_id': 'v_artist_2654', 'artwork_id': 'category030_1137', 'image_path': 'category030/category030_1137.jpg', 'matched_key': 'category030/category030_1137.jpg', 'clip_primary_label': 'category030', 'clip_primary_score': 0.285888671875}

LOG type: <class 'list'>
LOG len: 3708695
LOG[0] keys: ['member_id', 'artwork_id', 'timestamp']
LOG[0] sample: {'member_id': 'u_00000', 'artwork_id': 'category043_1420', 'timestamp': 'VIEW'}

item_mat: (27703, 512) num_items: 27702
num_users: 30000 missed_logs: 0
user seq len stats: min= 3 p50= 118 p90= 200 max= 200
len==0: 0 len==1: 0 len==2: 0 len>=3: 30000


In [14]:
# Cell 8) Load data (2만명 유저 + cold/normal/heavy stratified 80/20 split + 비율 출력)

artwork2idx, item_mat = load_item_vectors(VEC_PATH, expected_dim=CLIP_DIM)
user_item_seq, user_act_seq, missed = load_user_sequences(
    LOG_PATH, artwork2idx,
    logs_are_latest_first=LOGS_ARE_LATEST_FIRST,
    use_timestamp_sort=USE_TIMESTAMP_SORT
)

# 1) 유저 길이 기반 세그먼트 맵
user2len = {u: len(user_item_seq[u]) for u in user_item_seq.keys()}
user2seg = {u: segment_by_len(user2len[u]) for u in user_item_seq.keys()}

# 2) 학습/평가 가능 유저 필터
eligible = [u for u in user_item_seq.keys() if user2len[u] >= MIN_LEN_FOR_TRAIN]  # train용
if len(eligible) == 0:
    raise ValueError("No eligible users (len>=MIN_LEN_FOR_TRAIN). Check logs / parsing.")

# 3) 2만명 선택(옵션: 세그먼트 목표비율 enforce)
chosen = pick_users_with_optional_ratios(eligible, user2seg, NUM_USERS, seed=SEED)

# 4) stratified 80/20 split (cold/normal/heavy 분포 유지)
train_users, infer_users, chosen_counts = split_users_stratified(
    chosen, user2seg, train_ratio=TRAIN_USER_RATIO, seed=SEED
)

print(f"items={len(artwork2idx)-1} users_total={len(user_item_seq)} eligible_train={len(eligible)} chosen={len(chosen)}")
print(f"train_users={len(train_users)} infer_users={len(infer_users)} missed_logs={missed}")
print("item_mat:", tuple(item_mat.shape))

# 분포 출력(요청한 “비율 유지/맞춘 비중” 확인용)
chosen_c, chosen_r = print_split_stats("CHOSEN", chosen, user2seg)
train_c, train_r   = print_split_stats("TRAIN", train_users, user2seg)
infer_c, infer_r   = print_split_stats("INFER", infer_users, user2seg)

if ENFORCE_SEGMENT_RATIOS:
    print("[TARGET] cold/normal/heavy ratios:",
          f"{TARGET_COLD_RATIO:.3f}/{TARGET_NORMAL_RATIO:.3f}/{TARGET_HEAVY_RATIO:.3f}")

# 5) dict로 분리
train_item = {u: user_item_seq[u] for u in train_users}
train_act  = {u: user_act_seq[u]  for u in train_users}
infer_item = {u: user_item_seq[u] for u in infer_users}
infer_act  = {u: user_act_seq[u]  for u in infer_users}

# 6) dataset/loader
dataset = SASRecDataset(train_item, train_act, maxlen=MAX_LEN, exclude_last_target=False)
print("train samples:", len(dataset))

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

items=27702 users_total=30000 eligible_train=30000 chosen=20000
train_users=16000 infer_users=4000 missed_logs=0
item_mat: (27703, 512)
[CHOSEN] n=20000 | cold=286 (0.014) normal=1805 (0.090) heavy=17909 (0.895)
[TRAIN] n=16000 | cold=229 (0.014) normal=1444 (0.090) heavy=14327 (0.895)
[INFER] n=4000 | cold=57 (0.014) normal=361 (0.090) heavy=3582 (0.895)
train samples: 16000


In [15]:
# Cell 9) Build models + optimizer

# ✅ item vectors는 런타임 주입이므로, device로 한 번만 올려둔다 (학습 파라미터 아님)
item_mat = item_mat.to(device)

sas_model = VectorSASRec(
    clip_dim=CLIP_DIM,
    hidden_dim=HIDDEN_DIM,
    num_actions=N_ACTIONS,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    dropout=DROPOUT,
    maxlen=MAX_LEN,
).to(device)

tt_model = TwoTowerAlign(dim=HIDDEN_DIM, dropout=DROPOUT).to(device)

params = list(sas_model.parameters()) + list(tt_model.parameters())
optimizer = torch.optim.Adam(params, lr=LR)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

best_hr10 = -1.0
best_hr20 = -1.0


/home/j-i14e107/.conda/envs/ai_dev_env/lib/python3.9/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [16]:
# Cell 10) Train loop + MLflow logging + 세그먼트별 결과 출력 (FIXED)

import os, time

def safe_mlflow_call(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("[MLFLOW WARN]", type(e).__name__, str(e)[:200], flush=True)
        return None

def _mlflow_setup():
    safe_mlflow_call(mlflow.set_tracking_uri, MLFLOW_TRACKING_URI)
    safe_mlflow_call(mlflow.set_experiment, MLFLOW_EXPERIMENT)

def _log_split_to_mlflow(prefix: str, counts: dict, ratios: dict):
    for seg in ["cold", "normal", "heavy"]:
        safe_mlflow_call(mlflow.log_metric, f"{prefix}_count_{seg}", int(counts.get(seg, 0)))
        safe_mlflow_call(mlflow.log_metric, f"{prefix}_ratio_{seg}", float(ratios.get(seg, 0.0)))

# --- MLflow start (안되면 꺼버리고 학습 계속) ---
if MLFLOW_ON:
    _mlflow_setup()
    run = safe_mlflow_call(mlflow.start_run, run_name=MLFLOW_RUN_NAME)
    if run is None:
        print("[MLFLOW] start_run failed -> disable mlflow for this run", flush=True)
        MLFLOW_ON = False

try:
    if MLFLOW_ON:
        # params
        safe_mlflow_call(mlflow.log_params, {
            "hidden_dim": HIDDEN_DIM,
            "max_len": MAX_LEN,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "lr": LR,
            "seed": SEED,
            "n_layers": N_LAYERS,
            "n_heads": N_HEADS,
            "dropout": DROPOUT,
            "logit_scale": LOGIT_SCALE,
            "num_actions": N_ACTIONS,
            "num_users_chosen": len(chosen),
            "train_user_ratio": TRAIN_USER_RATIO,
            "cold_max_len": COLD_MAX_LEN,
            "normal_max_len": NORMAL_MAX_LEN,
            "use_timestamp_sort": USE_TIMESTAMP_SORT,
            "enforce_segment_ratios": ENFORCE_SEGMENT_RATIOS,
            "target_cold_ratio": TARGET_COLD_RATIO,
            "target_normal_ratio": TARGET_NORMAL_RATIO,
            "target_heavy_ratio": TARGET_HEAVY_RATIO,
        })
        # split ratios
        _log_split_to_mlflow("chosen", chosen_c, chosen_r)
        _log_split_to_mlflow("train_users", train_c, train_r)
        _log_split_to_mlflow("infer_users", infer_c, infer_r)

    # --- training ---
    best_hr10 = -1.0
    best_hr20 = -1.0

    HB_EVERY = 50  # heartbeat every N steps

    for epoch in range(1, EPOCHS + 1):
        sas_model.train()
        tt_model.train()
        total_loss = 0.0
        n_steps = 0

        t_epoch0 = time.time()
        pbar = tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=True)

        for input_items, input_acts, target_items in pbar:
            input_items  = input_items.to(device)
            input_acts   = input_acts.to(device)
            target_items = target_items.to(device)

            sas_emb = sas_model(input_items, input_acts, item_mat)  # (B,S,H)
            B, S, H = sas_emb.shape

            targets = target_items.reshape(-1)  # (B*S)
            mask = targets != 0
            if mask.sum().item() == 0:
                continue

            user_final = tt_model.user_proj(sas_emb.reshape(-1, H))
            user_final = user_final[mask]
            user_final = F.normalize(user_final, p=2, dim=-1)

            item_final = tt_model.item_proj(sas_model.item_base(item_mat))
            item_final = F.normalize(item_final, p=2, dim=-1)

            logits = torch.matmul(user_final, item_final.T)
            logits[:, 0] = -1e9
            logits = logits * LOGIT_SCALE

            loss = loss_fn(logits, targets[mask])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += float(loss.item())
            n_steps += 1
            pbar.set_postfix(loss=float(loss.item()))

            # heartbeat
            if (n_steps % HB_EVERY) == 0:
                msg = f"[HB] epoch={epoch} step={n_steps} loss={float(loss.item()):.4f} elapsed={((time.time()-t_epoch0)/60):.1f}m"
                if torch.cuda.is_available():
                    alloc = torch.cuda.memory_allocated() / (1024**3)
                    rsv = torch.cuda.memory_reserved() / (1024**3)
                    msg += f" | cuda_alloc={alloc:.2f}GB cuda_rsv={rsv:.2f}GB"
                print(msg, flush=True)

        avg_loss = total_loss / max(n_steps, 1)
        print(f"[Epoch {epoch}] avg_loss={avg_loss:.4f} | evaluating(infer users)...", flush=True)

        # 전체(infer 전체) 평가
        hr10, hr20, ndcg10, ndcg20 = evaluate_valid_action(
            sas_model, tt_model,
            infer_item, infer_act,
            item_mat.detach(),
            maxlen=MAX_LEN,
            device=device
        )
        print(f"   [ALL] HR@10={hr10:.4f} HR@20={hr20:.4f} NDCG@10={ndcg10:.4f} NDCG@20={ndcg20:.4f}", flush=True)

        # 세그먼트별 평가
        seg_metrics = evaluate_by_segment(
            sas_model, tt_model,
            infer_item, infer_act,
            user2seg,
            item_mat.detach(),
            maxlen=MAX_LEN,
            device=device
        )
        for seg in ["cold", "normal", "heavy"]:
            m = seg_metrics[seg]
            print(f"   [{seg.upper():6s}] n={m['n_users']:5d} | "
                  f"HR@10={m['hr10']:.4f} HR@20={m['hr20']:.4f} "
                  f"NDCG@10={m['ndcg10']:.4f} NDCG@20={m['ndcg20']:.4f}", flush=True)

        # MLflow log
        if MLFLOW_ON:
            safe_mlflow_call(mlflow.log_metric, "train_avg_loss", avg_loss, step=epoch)

            safe_mlflow_call(mlflow.log_metric, "eval_all_hr10", hr10, step=epoch)
            safe_mlflow_call(mlflow.log_metric, "eval_all_hr20", hr20, step=epoch)
            safe_mlflow_call(mlflow.log_metric, "eval_all_ndcg10", ndcg10, step=epoch)
            safe_mlflow_call(mlflow.log_metric, "eval_all_ndcg20", ndcg20, step=epoch)

            for seg in ["cold", "normal", "heavy"]:
                m = seg_metrics[seg]
                safe_mlflow_call(mlflow.log_metric, f"eval_{seg}_hr10", m["hr10"], step=epoch)
                safe_mlflow_call(mlflow.log_metric, f"eval_{seg}_hr20", m["hr20"], step=epoch)
                safe_mlflow_call(mlflow.log_metric, f"eval_{seg}_ndcg10", m["ndcg10"], step=epoch)
                safe_mlflow_call(mlflow.log_metric, f"eval_{seg}_ndcg20", m["ndcg20"], step=epoch)
                safe_mlflow_call(mlflow.log_metric, f"eval_{seg}_n_users", m["n_users"], step=epoch)

        # best 저장 기준: HR@20
        if hr20 > best_hr20:
            best_hr20 = hr20
            best_hr10 = max(best_hr10, hr10)
            print(f"   [Best] saving checkpoints (best_hr20={best_hr20:.4f})", flush=True)

            torch.save(
                {"state_dict": sas_model.state_dict(),
                 "config": {"model_type": "VectorSASRec_v1", "clip_dim": CLIP_DIM, "num_actions": N_ACTIONS, "hidden": HIDDEN_DIM, "maxlen": MAX_LEN,
                            "n_layers": N_LAYERS, "n_heads": N_HEADS, "dropout": DROPOUT,
                            "num_actions": N_ACTIONS,
                            "cold_max_len": COLD_MAX_LEN, "normal_max_len": NORMAL_MAX_LEN}},
                OUT_SASREC
            )
            torch.save(
                {"two_tower_state_dict": tt_model.state_dict(),
                 "config": {"hidden": HIDDEN_DIM, "dropout": DROPOUT}},
                OUT_TWOTOWER
            )

            # ✅ artifact 기록(오타 제거)
            if MLFLOW_ON:
                if os.path.exists(OUT_SASREC):
                    safe_mlflow_call(mlflow.log_artifact, OUT_SASREC, artifact_path="checkpoints")
                if os.path.exists(OUT_TWOTOWER):
                    safe_mlflow_call(mlflow.log_artifact, OUT_TWOTOWER, artifact_path="checkpoints")

    print("\nDone. best_hr20:", best_hr20, "| best_hr10:", best_hr10, flush=True)

    print("\n==== FINAL SPLIT RATIOS (CHOSEN/TRAIN/INFER) ====", flush=True)
    print_split_stats("CHOSEN", chosen, user2seg)
    print_split_stats("TRAIN", train_users, user2seg)
    print_split_stats("INFER", infer_users, user2seg)

    if MLFLOW_ON:
        if os.path.exists(OUT_SASREC):
            safe_mlflow_call(mlflow.log_artifact, OUT_SASREC, artifact_path="checkpoints")
        if os.path.exists(OUT_TWOTOWER):
            safe_mlflow_call(mlflow.log_artifact, OUT_TWOTOWER, artifact_path="checkpoints")

        safe_mlflow_call(mlflow.log_metric, "best_hr20", best_hr20)
        safe_mlflow_call(mlflow.log_metric, "best_hr10", best_hr10)

finally:
    if MLFLOW_ON:
        safe_mlflow_call(mlflow.end_run)

Epoch 1/30:  78%|███████▊  | 49/63 [00:18<00:03,  4.02it/s, loss=9.49]

[HB] epoch=1 step=50 loss=9.4898 elapsed=0.3m | cuda_alloc=3.60GB cuda_rsv=38.40GB


Epoch 1/30: 100%|██████████| 63/63 [00:21<00:00,  2.92it/s, loss=9.28]

[Epoch 1] avg_loss=9.5497 | evaluating(infer users)...


   [ALL] HR@10=0.1985 HR@20=0.2260 NDCG@10=0.1225 NDCG@20=0.1297


...

Epoch 30/30: 100%|██████████| 63/63 [00:15<00:00,  4.03it/s, loss=8.66]

[Epoch 30] avg_loss=8.6703 | evaluating(infer users)...


   [ALL] HR@10=0.2028 HR@20=0.2417 NDCG@10=0.1276 NDCG@20=0.1374


   [COLD  ] n=   57 | HR@10=0.2456 HR@20=0.2982 NDCG@10=0.1390 NDCG@20=0.1532
   [NORMAL] n=  361 | HR@10=0.2355 HR@20=0.2770 NDCG@10=0.1458 NDCG@20=0.1565
   [HEAVY ] n= 3582 | HR@10=0.1988 HR@20=0.2373 NDCG@10=0.1256 NDCG@20=0.1352



Done. best_hr20: 0.24175 | best_hr10: 0.20025

==== FINAL SPLIT RATIOS (CHOSEN/TRAIN/INFER) ====
[CHOSEN] n=20000 | cold=286 (0.014) normal=1805 (0.090) heavy=17909 (0.895)
[TRAIN] n=16000 | cold=229 (0.014) normal=1444 (0.090) heavy=14327 (0.895)
[INFER] n=4000 | cold=57 (0.014) normal=361 (0.090) heavy=3582 (0.895)
